# 토큰 트리밍 & 요약 메모리 — Modern 패턴

## 이번 노트북 학습 목표

- 글자 수(문자 수)를 기준으로 히스토리를 잘라내는 **토큰 트리밍 패턴**을 구현해요.
- LLM으로 오래된 대화를 압축하는 **요약 기반 메모리 패턴**을 구현해요.
- 두 패턴의 차이와 적합한 사용 시나리오를 이해해요.

## 사전 지식

- `ChatMessageHistory` 기본 사용법 (10번 노트북)
- Modern 윈도우 메모리 패턴 (12번 노트북)

## 구식 vs Modern 메모리 패턴 비교

| 항목 | 구식 (03번, `ConversationTokenBufferMemory`) | Modern 토큰 트리밍 |
|------|----------------------------------------------|-------------------|
| 메모리 클래스 | `ConversationTokenBufferMemory(max_token_limit=500)` | 없음 (함수로 대체) |
| 구현 방식 | 클래스 내부에서 자동 트리밍 | `trim_messages_by_chars()` 함수로 명시적 처리 |
| LangChain 권장 | 0.2.x 이후 deprecated | 1.0.x 권장 방식 |

| 항목 | 구식 (06번, `ConversationSummaryMemory`) | Modern 요약 메모리 |
|------|------------------------------------------|-------------------|
| 메모리 클래스 | `ConversationSummaryMemory(llm=...)` | 없음 (요약 체인으로 대체) |
| 구현 방식 | 클래스가 자동으로 요약 | `summarize_session()` 함수로 명시적 처리 |
| LangChain 권장 | 0.2.x 이후 deprecated | 1.0.x 권장 방식 |

> 🎯 **강의 포인트**: 토큰 트리밍은 오래된 메시지를 **삭제**하고, 요약 메모리는 오래된 메시지를 **압축 보존**합니다. 요약 메모리가 맥락 손실은 적지만, 매 턴마다 LLM 추가 호출 비용이 발생합니다.

> ⚠️ **자주 하는 실수**: 요약 기반 패턴에서 `summarize_session()`은 매 턴마다 LLM을 추가 호출합니다. N턴마다 1회 요약하도록 조절하면 비용을 크게 줄일 수 있습니다.

> 이 노트북에서는 이해를 돕기 위해 **문자 수 기준**으로 구현해요. 실제 서비스에서는 `get_num_tokens()` 등 토큰 계산 유틸이나 LangGraph의 `SummarizationMiddleware`를 사용하는 것이 좋아요.

## 언제 어떤 패턴을 선택해야 할까?

메모리 패턴은 서비스 특성에 따라 선택해야 합니다. 아래 가이드를 참고하세요.

### 패턴별 적합한 사용 시나리오

| 패턴 | 적합한 상황 | 실전 예시 |
|------|------------|----------|
| **토큰 트리밍** | 단기 세션, 최근 대화만 필요 | 고객 문의 응대 ("배송 어디예요?") |
| **요약 메모리** | 장기 세션, 초기 맥락이 중요 | 프로젝트 컨설팅 (한 달간 진행 상황 추적) |
| **하이브리드** | 둘 다 필요한 복합 상황 | 진료 상담 (최근 증상은 원문 + 과거 병력은 요약) |

### 선택 플로차트

```mermaid
flowchart TD
    START[대화 세션이 길어질 수 있나요?] -- 아니오 --> TRIM[토큰 트리밍 패턴<br/>최근 대화만 유지하면 충분]
    START -- 예 --> CONTEXT[과거 대화의 세부 내용이 중요한가요?]
    CONTEXT -- 아니오 --> SUMMARY[요약 메모리 패턴<br/>과거를 압축해서 보존]
    CONTEXT -- 예 --> HYBRID[하이브리드 패턴<br/>최근 원문 + 과거 요약]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class START input
    class TRIM,SUMMARY,HYBRID output
    class CONTEXT process
```

### 비용 vs 품질 트레이드오프

| | 토큰 비용 | 맥락 보존 | 구현 난이도 | LLM 추가 호출 |
|---|---|---|---|---|
| 토큰 트리밍 | 낮음 | 낮음 (오래된 것 삭제) | 쉬움 | 없음 |
| 요약 메모리 | 중간 | 높음 (압축 보존) | 보통 | 매 턴마다 1회 |
| 하이브리드 | 중간 | 매우 높음 | 높음 | 매 턴마다 1회 |

In [1]:
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

load_dotenv()

# LLM 초기화
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 기본 프롬프트 (전체 히스토리 사용)
base_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 기술 지원 상담원입니다. 사용자의 문제를 차분히 도와주세요."),
        MessagesPlaceholder("chat_history"),
        ("human", "{question}"),
    ]
)

base_chain = base_prompt | model | StrOutputParser()

# 세션별 대화 기록 저장소
store: dict[str, ChatMessageHistory] = {}


def get_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


## 1. 글자 수 기준 토큰 트리밍 패턴

`ConversationTokenBufferMemory(max_token_limit=...)`(구식 토큰 버퍼 메모리)와 유사하게, **히스토리의 총 길이가 한계를 넘으면 가장 오래된 메시지부터 제거**하는 패턴이에요.

여기서는 이해를 돕기 위해 실제 토큰 수 대신 **문자 수(글자 수)**를 기준으로 구현해요.

### 토큰 트리밍 흐름

```mermaid
flowchart TD
    QUESTION[사용자 질문] --> CHAIN[char_limited_chain 실행]
    CHAIN --> HIST[history.messages 전체 조회]
    HIST --> CHECK{"총 문자 수 <= max_chars?"}
    CHECK -- 예 --> USE[그대로 사용]
    CHECK -- 아니오 --> TRIM[가장 오래된 메시지 제거]
    TRIM --> CHECK
    USE --> PROMPT["base_prompt.invoke<br/>트리밍된 히스토리 주입"]
    PROMPT --> LLM[ChatOpenAI]
    LLM --> SAVE[세션 히스토리 저장<br/>원본 보존]
    SAVE --> ANSWER[AI 응답]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef external fill:#e2e3e5,stroke:#6c757d,color:#383d41

    class QUESTION input
    class CHAIN,CHECK,TRIM,USE,PROMPT,LLM process
    class HIST,SAVE storage
    class ANSWER output
```

> 히스토리 원본은 모두 보관하지만, 프롬프트에 들어가는 메시지는 `max_chars` 이내로 제한해요. 오래된 정보가 완전히 사라지는 단점이 있어요. 장기 맥락이 필요하다면 아래 요약 패턴을 사용하세요.

In [2]:
# ============================================================
# TODO: 문자 수 기반 트리밍 함수와 제한 체인을 구현하세요
# 힌트: total_chars()로 전체 길이 계산 후, max_chars를 초과하면 앞에서부터 제거하세요
# 예상 결과: 오래된 대화가 제거되어 max_chars 이내로 유지되어야 합니다
# ============================================================

# ---------------------------------------------------
# 글자 수 기반 트리밍 함수 및 체인 구현
# ---------------------------------------------------

from typing import List
from langchain_core.messages import BaseMessage


def total_chars(messages: List[BaseMessage]) -> int:
    """메시지 리스트의 총 문자 수를 계산."""
    # 각 메시지 내용의 길이를 합산
    return sum(len(m.content) for m in messages)


def trim_messages_by_chars(messages: List[BaseMessage], max_chars: int) -> List[BaseMessage]:
    """총 문자 수가 max_chars를 넘지 않도록 앞에서부터 메시지를 제거.

    - 시스템 메시지 1개 + 최근 대화 몇 개만 남기는 간단한 구현
    """
    # 이미 한도 내에 있으면 그대로 반환
    if total_chars(messages) <= max_chars:
        return messages

    trimmed = list(messages)
    while trimmed and total_chars(trimmed) > max_chars:
        # 가장 오래된 메시지 하나 제거 (앞에서부터 pop)
        trimmed.pop(0)
    return trimmed


def token_like_chain(max_chars: int = 4000):
    """문자 수 한계를 가진 토큰 버퍼 체인을 생성."""

    def _chain(inputs: dict) -> str:
        session_id: str = inputs["session_id"]
        question: str = inputs["question"]

        history = get_history(session_id)

        # 1) 현재까지의 전체 메시지
        all_messages = history.messages

        # 2) 문자 수 기준으로 트리밍 (max_chars 초과 시 오래된 것 제거)
        trimmed = trim_messages_by_chars(all_messages, max_chars=max_chars)

        # 3) 트리밍된 히스토리 + 현재 질문으로 프롬프트 구성
        prompt_msg = base_prompt.invoke({
            "chat_history": trimmed,
            "question": question,
        })

        # 4) 모델 호출
        result = (model | StrOutputParser()).invoke(prompt_msg)

        # 5) 실제 히스토리에는 전체 대화를 계속 쌓음 (원본 보존)
        history.add_user_message(question)
        history.add_ai_message(result)

        return result

    return _chain


# max_chars=600: 약 600자 초과 시 오래된 메시지부터 제거
char_limited_chain = token_like_chain(max_chars=600)

In [3]:
# ---------------------------------------------------
# 토큰 유사 트리밍 체인 테스트: 오래된 대화가 잊혀지는지 확인
# ---------------------------------------------------

tech_session = "tech_support_token_limited"

questions = [
    "안녕하세요. 소프트웨어 설치 중에 에러 코드 1722가 발생했어요.",
    "에러가 나는 단계는 설치 마법사 마지막 단계입니다.",
    "로그를 어디서 확인해야 하는지도 알려주세요.",
    "어제 이야기했던 방화벽 설정 관련 내용도 다시 설명해 주실 수 있나요?",
    "지금까지 제가 어떤 환경이라고 설명했는지도 기억하시나요?",
]

print("=" * 60)
print("💾 문자 수 제한 기반 토큰 버퍼 메모리")
print("=" * 60)

# 각 질문 실행 후 응답 출력 (max_chars 초과 시 오래된 내용은 잊혀짐)
for i, q in enumerate(questions, start=1):
    answer = char_limited_chain({"session_id": tech_session, "question": q})
    print(f"{i}. [사용자] {q}")
    # 응답이 길면 앞부분만 표시
    print(f"   [AI] {answer[:120]}...\n")

💾 문자 수 제한 기반 토큰 버퍼 메모리
1. [사용자] 안녕하세요. 소프트웨어 설치 중에 에러 코드 1722가 발생했어요.
   [AI] 안녕하세요! 에러 코드 1722는 일반적으로 Windows Installer와 관련된 문제로 발생합니다. 이 오류는 설치 프로그램이 특정 작업을 완료하지 못했음을 나타냅니다. 다음은 이 문제를 해결하기 위한 몇 가지...

2. [사용자] 에러가 나는 단계는 설치 마법사 마지막 단계입니다.
   [AI] 설치 마법사의 마지막 단계에서 에러가 발생하는 경우, 몇 가지 점검해볼 사항이 있습니다. 다음 단계를 따라해 보세요:

1. **에러 메시지 확인**: 에러 메시지가 나타난다면, 그 내용을 메모해 두세요. 특정 에러...

3. [사용자] 로그를 어디서 확인해야 하는지도 알려주세요.
   [AI] 로그를 확인하는 방법은 사용하는 시스템이나 소프트웨어에 따라 다를 수 있습니다. 일반적으로 다음과 같은 방법으로 로그를 확인할 수 있습니다:

1. **운영 체제 로그**:
   - **Windows**: 이벤트 뷰...

4. [사용자] 어제 이야기했던 방화벽 설정 관련 내용도 다시 설명해 주실 수 있나요?
   [AI] 물론입니다! 방화벽 설정에 대해 다시 설명해 드리겠습니다.

방화벽은 네트워크의 보안을 강화하기 위해 들어오는 트래픽과 나가는 트래픽을 제어하는 장치입니다. 방화벽 설정은 일반적으로 다음과 같은 단계로 진행됩니다:
...

5. [사용자] 지금까지 제가 어떤 환경이라고 설명했는지도 기억하시나요?
   [AI] 죄송하지만, 이전 대화 내용을 기억할 수는 없습니다. 하지만 현재의 환경이나 문제에 대해 설명해 주시면 최선을 다해 도와드리겠습니다. 어떤 도움이 필요하신가요?...



## 메모리 패턴별 동작 비교: 20턴 대화 시 무슨 일이 벌어질까?

동일한 20턴 대화를 세 가지 메모리 패턴으로 처리했을 때의 차이를 비교해 봅시다.

### 1) 전체 버퍼 메모리 (BufferMemory) - 20턴 후

```
프롬프트 크기: ████████████████████████████████████████ (계속 증가!)
턴 1 내용: 기억함
턴 10 내용: 기억함
턴 20 내용: 기억함
토큰 비용: 매우 높음 (20턴 전체가 매번 프롬프트에 포함)
문제점: 토큰 한계를 초과하면 에러 발생!
```

### 2) 토큰 트리밍 (이 노트북 Section 1) - 20턴 후

```
프롬프트 크기: ████████ (일정하게 유지)
턴 1 내용: 잊어버림 (삭제됨)
턴 10 내용: 잊어버림 (삭제됨)
턴 20 내용: 기억함 (최근 몇 턴만 유지)
토큰 비용: 낮음 (항상 max_chars 이내)
문제점: 초반 중요 정보가 영구 삭제됨
```

### 3) 요약 메모리 (이 노트북 Section 2) - 20턴 후

```
프롬프트 크기: ██████████ (일정 + 요약 크기)
턴 1 내용: 요약에 포함 (압축 보존)
턴 10 내용: 요약에 포함 (압축 보존)
턴 20 내용: 최근 원문으로 유지
토큰 비용: 중간 (요약 LLM 호출 비용 추가)
장점: 과거 맥락을 잃지 않으면서도 프롬프트 크기 제어 가능
```

### 핵심 차이 요약

| 상황 | 전체 버퍼 | 토큰 트리밍 | 요약 메모리 |
|------|----------|-----------|-----------|
| "첫 번째 대화에서 뭐라고 했죠?" | 정확히 답변 | "기억 못합니다" | 요약 기반으로 대략 답변 |
| 토큰 비용 (20턴 기준) | ~40,000 토큰 | ~4,000 토큰 | ~6,000 토큰 (요약 호출 포함) |
| 100턴 이상 대화 | 토큰 한계 초과 에러 | 정상 동작 (최근만) | 정상 동작 (요약 축적) |

## 2. 요약 기반 메모리 패턴

`ConversationSummaryMemory(llm=...)`(구식 요약 메모리)는 클래스가 자동으로 요약을 관리했어요. Modern 패턴에서는 **요약 체인(summary chain)**을 직접 만들고, `summary_store`에 텍스트로 저장해요.

핵심 아이디어:
- 세션별로 `summary_store[session_id]`에 **지금까지의 대화 요약 텍스트**를 저장해요.
- 새 질문이 들어올 때마다 `old_summary + recent_history → LLM → new_summary`로 갱신해요.
- 프롬프트에는 "과거 요약 + 최근 몇 턴"을 함께 넣어요.

### 요약 기반 메모리 흐름

```mermaid
flowchart TD
    QUESTION[사용자 질문] --> HADD[history.add_user_message]
    HADD --> SUM_FN[summarize_session<br/>old_summary + recent 메시지]
    SUM_FN --> SUM_LLM[요약 체인<br/>summary_chain]
    SUM_LLM --> SUM_STORE[summary_store 갱신]
    SUM_STORE --> PROMPT[프롬프트 구성<br/>과거 요약 + 최근 대화 + 질문]
    PROMPT --> LLM[ChatOpenAI]
    LLM --> HADD2[history.add_ai_message]
    HADD2 --> ANSWER[AI 응답]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef external fill:#e2e3e5,stroke:#6c757d,color:#383d41

    class QUESTION input
    class HADD,SUM_FN,SUM_LLM,PROMPT,LLM,HADD2 process
    class SUM_STORE storage
    class ANSWER output
```

> 요약을 생성하는 LLM 호출이 **추가로 발생**해요. 매 턴마다 요약 비용이 생기지만, 덕분에 히스토리가 길어져도 프롬프트 크기는 일정하게 유지돼요.

In [4]:
# ============================================================
# TODO: 요약 기반 메모리 체인을 구현하세요
# 힌트: 매 턴마다 summary_chain으로 과거 요약을 갱신하고, 프롬프트에 요약을 주입하세요
# 예상 결과: 마지막 질문에서 AI가 초반 대화 내용을 요약으로 기억해야 합니다
# ============================================================

# ---------------------------------------------------
# 요약 기반 메모리 구현: 요약 체인 + 스토어 + 메모리 체인
# ---------------------------------------------------

# 1단계: 요약 생성에 사용할 프롬프트
# - old_summary: 이전 요약 (없으면 "(아직 요약 없음)")
# - recent_history: 최근 대화 텍스트
summary_prompt = ChatPromptTemplate.from_template(
    """
    아래는 지금까지의 대화 기록입니다.

    --- 과거 요약 ---
    {old_summary}
    -----------------

    --- 최근 대화 ---
    {recent_history}
    -----------------

    위 정보를 바탕으로, 사용자의 상황과 진행 상황을 5줄 이내의 한국어 요약으로 정리해 주세요.
    """
)

# summary_chain: 요약 생성 전용 체인 (매 턴마다 호출)
# 💡 팁: 매 턴마다 LLM 호출이 추가 발생합니다. N턴마다 1회 요약하도록 조절하면 비용을 절감할 수 있습니다
summary_chain = summary_prompt | model | StrOutputParser()

# 세션별 요약 저장소 (키: session_id, 값: 요약 텍스트)
summary_store: dict[str, str] = {}


def format_messages(messages: list[BaseMessage]) -> str:
    """메시지 리스트를 사람이 읽기 좋은 문자열로 변환."""
    lines = []
    for m in messages:
        role = "사용자" if m.type in ("human", "user") else "AI"
        lines.append(f"[{role}] {m.content}")
    return "\n".join(lines)


def summarize_session(session_id: str, max_recent: int = 6) -> str:
    """세션의 최근 메시지를 기존 요약과 합쳐 새로운 요약 생성."""
    history = get_history(session_id)
    # 최근 max_recent개 메시지를 텍스트로 변환
    recent_msgs = history.messages[-max_recent:]
    recent_text = format_messages(recent_msgs)

    # 기존 요약이 없으면 빈 요약으로 시작
    old_summary = summary_store.get(session_id, "(아직 요약 없음)")

    # LLM으로 새 요약 생성 (과거 요약 + 최근 대화 → 압축)
    new_summary = summary_chain.invoke({
        "old_summary": old_summary,
        "recent_history": recent_text,
    })

    # 갱신된 요약을 스토어에 저장
    summary_store[session_id] = new_summary
    return new_summary


def summary_memory_chain(max_recent: int = 6):
    """요약 + 최근 히스토리를 함께 사용하는 체인."""

    def _chain(inputs: dict) -> str:
        session_id: str = inputs["session_id"]
        question: str = inputs["question"]

        history = get_history(session_id)

        # 1) 질문/답변을 히스토리에 먼저 추가
        history.add_user_message(question)

        # 2) 요약 업데이트 (과거 요약 + 최근 max_recent 메시지)
        # ⚠️ 주의: 이 LLM 호출이 매 턴마다 추가로 발생합니다 (비용 주의)
        current_summary = summarize_session(session_id, max_recent=max_recent)

        # 3) 프롬프트를 위해 "요약 + 최근 대화" 구성
        recent_msgs = history.messages[-max_recent:]
        recent_text = format_messages(recent_msgs)

        prompt_msg = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "당신은 긴 대화를 요약해 기억하는 기술 지원 상담원입니다.\n"
                    "과거 요약과 최근 대화만을 사용해 답변하세요.",
                ),
                (
                    "system",
                    "[과거 요약]\n" "{summary}\n\n[최근 대화]\n" "{recent}\n",
                ),
                ("human", "{question}"),
            ]
        ).invoke(
            {
                "summary": current_summary,
                "recent": recent_text,
                "question": question,
            }
        )

        result = (model | StrOutputParser()).invoke(prompt_msg)

        # 4) 히스토리에 AI 응답 추가
        history.add_ai_message(result)

        return result

    return _chain


summary_chain_with_memory = summary_memory_chain(max_recent=6)

In [5]:
# ---------------------------------------------------
# 요약 메모리 체인 테스트: 긴 대화 후 요약 활용 확인
# ---------------------------------------------------

summary_session = "tech_support_summary"

summary_questions = [
    "안녕하세요. 회사 노트북이 부팅이 안 돼서 문의드립니다.",
    "어제부터 전원 버튼을 눌러도 로고만 나오고 멈춰요.",
    "어떤 진단 절차를 먼저 진행해야 할까요?",
    "어제 알려주신 BIOS 초기화 방법을 다시 한 번 설명해 주세요.",
    "지금까지 제 장비 환경과 시도했던 조치들을 요약해서 말씀해 주실 수 있나요?",
]

print("=" * 60)
print("📝 요약 기반 메모리")
print("=" * 60)

# 매 턴마다 요약 체인이 호출되어 summary_store가 갱신됨
for i, q in enumerate(summary_questions, start=1):
    answer = summary_chain_with_memory({"session_id": summary_session, "question": q})
    print(f"{i}. [사용자] {q}")
    print(f"   [AI] {answer[:120]}...\n")

# 최종 저장된 요약 확인
print("\n현재 저장된 요약:")
print("-" * 60)
print(summary_store[summary_session])

📝 요약 기반 메모리
1. [사용자] 안녕하세요. 회사 노트북이 부팅이 안 돼서 문의드립니다.
   [AI] 안녕하세요! 회사 노트북의 부팅 문제에 대해 도움을 드리겠습니다. 현재 노트북이 어떤 상태인지, 예를 들어 화면에 어떤 메시지가 나타나는지, 또는 어떤 소리가 나는지 등의 추가 정보를 제공해 주실 수 있나요?...

2. [사용자] 어제부터 전원 버튼을 눌러도 로고만 나오고 멈춰요.
   [AI] 현재 노트북이 전원 버튼을 눌렀을 때 로고만 나타나고 멈추는 상태라는 점을 확인했습니다. 이 문제를 해결하기 위해 몇 가지 방법을 시도해 보겠습니다. 

1. **하드 리셋**: 노트북의 전원을 끄고, 배터리를 제거...

3. [사용자] 어떤 진단 절차를 먼저 진행해야 할까요?
   [AI] 먼저 **하드 리셋**을 시도해 보시는 것이 좋습니다. 노트북의 전원을 끄고 배터리를 제거한 후 몇 분간 기다렸다가 다시 연결한 다음 전원을 켜보세요. 이 방법이 효과가 없으면, 다음으로 **안전 모드 진입**을 시...

4. [사용자] 어제 알려주신 BIOS 초기화 방법을 다시 한 번 설명해 주세요.
   [AI] BIOS 초기화 방법은 다음과 같습니다:

1. **노트북 전원 끄기**: 먼저 노트북의 전원을 완전히 끕니다.

2. **BIOS 진입**: 전원을 켜고, 로고가 나타날 때 F2, DEL, ESC 또는 F10 키(...

5. [사용자] 지금까지 제 장비 환경과 시도했던 조치들을 요약해서 말씀해 주실 수 있나요?
   [AI] 현재 사용 중인 장비는 회사 노트북이며, 부팅 시 전원 버튼을 눌러도 로고만 나타나고 멈추는 상태입니다. 사용자는 다음과 같은 조치를 시도했습니다:

1. **하드 리셋**: 노트북의 전원을 끄고 배터리를 제거한 후...


현재 저장된 요약:
------------------------------------------------------------
사용자는 회사 노트북의 부팅 문제로 도움을 요청하고 있으며, 전원 버튼을 눌러도 

## 핵심 요약

### 이번 노트북의 구현 함수

| 함수 | 역할 |
|------|------|
| `trim_messages_by_chars(messages, max_chars)` | 총 문자 수가 한계를 초과하면 오래된 메시지를 앞에서부터 제거 |
| `token_like_chain(max_chars)` | 문자 수 제한 기반 체인 생성 함수 |
| `format_messages(messages)` | 메시지 리스트를 사람이 읽기 좋은 문자열로 변환 |
| `summarize_session(session_id, max_recent)` | 과거 요약 + 최근 메시지로 새 요약 생성 및 저장 |
| `summary_memory_chain(max_recent)` | 요약 기반 메모리 체인 생성 함수 |

### 두 패턴 비교

| 항목 | 토큰 트리밍 패턴 | 요약 기반 메모리 패턴 |
|------|-----------------|---------------------|
| 장기 맥락 유지 | 불가 (오래된 메시지 제거) | 가능 (요약으로 압축 보존) |
| 구현 복잡도 | 낮음 | 높음 |
| LLM 추가 호출 | 없음 | 매 턴마다 요약 호출 발생 |
| 비용 | 낮음 | 요약 호출 비용 추가 |
| 적합한 상황 | 단기 세션, 최신 대화만 필요할 때 | 장기 세션, 초기 맥락이 중요할 때 |

### 주의사항

- 이 노트북의 예제는 **문자 수 기준**으로 구현했어요. 실제 서비스에서는 `get_num_tokens()` 또는 `tiktoken` 라이브러리를 사용해 **실제 토큰 수**를 계산하세요.
- 요약 기반 패턴은 LLM 호출 비용이 추가로 발생해요. 요약 주기를 조절(예: N턴마다 1회)하면 비용을 줄일 수 있어요.
- 요약 과정에서 세부 정보가 일부 손실될 수 있어요. 법적·규정 맥락에서는 원본 히스토리를 별도 DB에 보관하세요.
- LangChain 1.0.x 이후에는 `LangGraph`의 `SummarizationMiddleware`가 이 패턴을 더 정교하게 지원해요.

### 하이브리드 패턴 (실전 권장)

실전에서는 두 방식을 **함께** 쓰는 것이 일반적이에요.

- 최근 N턴 원문 + 과거 요약 → 프롬프트에 함께 주입
- 나머지 생(raw) 히스토리는 별도 DB, 파일, 벡터 저장소(vector store)에 보관

### 다음 단계 예고

**14번**: `InMemoryStore`와 도구(Tool)를 사용해 특정 정보(사람 이름, 선호도 등)를 **구조화하여 추출·저장**하는 엔티티 메모리(entity memory) 패턴을 학습해요.

---

## 연습 문제

### 연습 1: 하이브리드 메모리 (요약 + 최근 대화) 구현

토큰 트리밍과 요약 메모리를 **결합**하여, "과거 대화는 요약으로 압축하고, 최근 대화는 원문으로 유지"하는 하이브리드 메모리 체인을 만들어 보세요.

**요구사항:**
1. 세션의 전체 대화 중 **최근 4개 메시지는 원문으로 유지**하고, 나머지 오래된 메시지는 **LLM으로 요약**하세요
2. 프롬프트에는 `[과거 요약]`과 `[최근 대화(원문)]`을 모두 포함하세요
3. 아래 시나리오로 테스트하세요 (총 6개 질문):
   - "안녕하세요. 저는 마케팅팀 이영희입니다."
   - "우리 팀은 다음 달에 신제품 런칭 캠페인을 준비하고 있어요."
   - "예산은 5천만원이고, 타깃은 2030 여성입니다."
   - "SNS 마케팅과 인플루언서 협업을 병행할 계획이에요."
   - "경쟁사 분석 결과도 공유드릴게요. A사는 가격, B사는 디자인이 강점이에요."
   - "지금까지 제가 공유한 캠페인 정보를 모두 요약해 주세요."
4. 마지막 질문에서 "과거 요약 + 최근 원문"을 모두 활용한 응답이 나오는지 확인하세요

**힌트:**
- 이미 노트북에 구현된 `summarize_session`과 `format_messages` 함수를 재활용하세요
- 프롬프트 구성 시 `system` 메시지에 요약 텍스트를, `MessagesPlaceholder`나 별도 변수에 최근 원문을 넣을 수 있습니다

In [6]:
# ============================================================
# TODO: 하이브리드 메모리 체인(요약 + 최근 원문)을 구현하세요
# 힌트: 최근 recent_count개는 원문으로, 나머지는 요약으로 처리하세요
# 예상 결과: 마지막 질문에서 초반 정보(요약)와 최근 정보(원문)를 모두 활용해야 합니다
# ============================================================

# ---------------------------------------------------
# 연습 1 풀이: 하이브리드 메모리 (요약 + 최근 원문) 구현
# ---------------------------------------------------

# 연습용 스토어 (기존과 분리)
exercise_store: dict[str, ChatMessageHistory] = {}
exercise_summary_store: dict[str, str] = {}


def get_exercise_history(session_id: str) -> ChatMessageHistory:
    if session_id not in exercise_store:
        exercise_store[session_id] = ChatMessageHistory()
    return exercise_store[session_id]


def exercise_summarize_old(session_id: str, recent_count: int = 4) -> str:
    """최근 recent_count개를 제외한 오래된 메시지를 요약."""
    history = get_exercise_history(session_id)
    all_msgs = history.messages

    # 오래된 메시지만 요약 대상 (최근 recent_count개는 제외)
    if len(all_msgs) <= recent_count:
        return exercise_summary_store.get(session_id, "(아직 요약할 오래된 대화가 없습니다.)")

    old_msgs = all_msgs[:-recent_count]
    old_text = format_messages(old_msgs)
    prev_summary = exercise_summary_store.get(session_id, "(없음)")

    # 기존 summary_chain 재사용 (노트북에서 이미 정의됨)
    new_summary = summary_chain.invoke({
        "old_summary": prev_summary,
        "recent_history": old_text,
    })
    exercise_summary_store[session_id] = new_summary
    return new_summary


def hybrid_memory_chain(recent_count: int = 4):
    """요약 + 최근 원문을 결합한 하이브리드 메모리 체인."""

    def _chain(inputs: dict) -> str:
        session_id: str = inputs["session_id"]
        question: str = inputs["question"]

        history = get_exercise_history(session_id)

        # 1) 사용자 질문을 히스토리에 추가
        history.add_user_message(question)

        # 2) 오래된 메시지 요약 (최근 recent_count개 제외한 나머지)
        old_summary = exercise_summarize_old(session_id, recent_count=recent_count)

        # 3) 최근 원문 메시지 가져오기
        recent_msgs = history.messages[-recent_count:]
        recent_text = format_messages(recent_msgs)

        # 4) 하이브리드 프롬프트 구성 (요약 + 최근 원문 + 현재 질문)
        hybrid_prompt = ChatPromptTemplate.from_messages([
            (
                "system",
                "당신은 마케팅 캠페인 기획을 돕는 AI 어시스턴트입니다.\n\n"
                "[과거 요약]\n{old_summary}\n\n"
                "[최근 대화(원문)]\n{recent_text}",
            ),
            ("human", "{question}"),
        ])

        prompt_msg = hybrid_prompt.invoke({
            "old_summary": old_summary,
            "recent_text": recent_text,
            "question": question,
        })

        result = (model | StrOutputParser()).invoke(prompt_msg)

        # 5) AI 응답 히스토리에 추가
        history.add_ai_message(result)

        return result

    return _chain


# 하이브리드 체인 생성 (최근 4개 메시지 원문 유지)
hybrid_chain = hybrid_memory_chain(recent_count=4)

# 마케팅 캠페인 시나리오 테스트
campaign_session = "campaign_hybrid"

campaign_questions = [
    "안녕하세요. 저는 마케팅팀 이영희입니다.",
    "우리 팀은 다음 달에 신제품 런칭 캠페인을 준비하고 있어요.",
    "예산은 5천만원이고, 타깃은 2030 여성입니다.",
    "SNS 마케팅과 인플루언서 협업을 병행할 계획이에요.",
    "경쟁사 분석 결과도 공유드릴게요. A사는 가격, B사는 디자인이 강점이에요.",
    "지금까지 제가 공유한 캠페인 정보를 모두 요약해 주세요.",
]

print("=" * 60)
print("[ 하이브리드 메모리 (요약 + 최근 원문) ]")
print("=" * 60)

for i, q in enumerate(campaign_questions, start=1):
    answer = hybrid_chain({"session_id": campaign_session, "question": q})
    print(f"{i}. [사용자] {q}")
    print(f"   [AI] {answer[:150]}...\n")

# 현재 저장된 요약 확인
print("=" * 60)
print("[ 저장된 과거 요약 ]")
print("=" * 60)
print(exercise_summary_store.get(campaign_session, "(없음)"))
print()
print("[ 최근 원문 메시지 수 ]")
history = get_exercise_history(campaign_session)
print(f"전체 메시지: {len(history.messages)}개")
print(f"최근 4개 원문으로 유지, 나머지는 요약으로 압축")

[ 하이브리드 메모리 (요약 + 최근 원문) ]
1. [사용자] 안녕하세요. 저는 마케팅팀 이영희입니다.
   [AI] 안녕하세요, 이영희님! 마케팅 캠페인 기획에 대해 어떤 도움을 드릴 수 있을까요? 필요한 정보나 아이디어가 있으시면 말씀해 주세요....

2. [사용자] 우리 팀은 다음 달에 신제품 런칭 캠페인을 준비하고 있어요.
   [AI] 신제품 런칭 캠페인을 준비하고 계시군요! 어떤 제품인지, 타겟 고객은 누구인지, 그리고 어떤 마케팅 채널을 고려하고 있는지에 대해 좀 더 자세히 말씀해 주시면, 캠페인 아이디어나 전략을 제안하는 데 도움이 될 것 같습니다....

3. [사용자] 예산은 5천만원이고, 타깃은 2030 여성입니다.
   [AI] 감사합니다, 이영희님! 2030 여성 타겟을 위한 신제품 런칭 캠페인에 대해 몇 가지 아이디어를 제안해 드리겠습니다.

1. **소셜 미디어 캠페인**: 인스타그램과 틱톡을 활용하여 제품을 홍보하는 캠페인을 진행할 수 있습니다. 인플루언서와 협업하여 제품 사용 후기나 ...

4. [사용자] SNS 마케팅과 인플루언서 협업을 병행할 계획이에요.
   [AI] 좋은 선택입니다, 이영희님! SNS 마케팅과 인플루언서 협업은 특히 2030 여성 타겟에게 효과적인 전략입니다. 다음은 이 두 가지를 효과적으로 결합하기 위한 몇 가지 구체적인 아이디어입니다:

1. **인플루언서 선정**: 타겟 고객층과 잘 맞는 인플루언서를 선정하세...

5. [사용자] 경쟁사 분석 결과도 공유드릴게요. A사는 가격, B사는 디자인이 강점이에요.
   [AI] 경쟁사 분석 결과를 공유해 주셔서 감사합니다, 이영희님! A사와 B사의 강점을 고려하여 신제품 런칭 캠페인 전략을 더욱 효과적으로 조정할 수 있을 것입니다. 다음은 이를 바탕으로 한 몇 가지 제안입니다:

1. **가격 경쟁력 강조**: A사가 가격이 강점이라면, 여러...

6. [사용자] 지금까지 제가 공유한 캠페인 정보를 모두 요약해 주세요.
   [AI] 이영희님이 공유한